In [ ]:
import mujoco
from mujoco import mjx
import jax.numpy as jnp
import jax
import os
import flax
from flax import linen as nn
from flax.training.train_state import TrainState
import optax

import pickle

mj_model = mujoco.MjModel.from_xml_path('bipedal.xml')

mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
mj_model.opt.iterations = 6
mj_model.opt.ls_iterations = 6

mjx_model = mjx.put_model(mj_model)
physics_steps_per_control_step = 5
ctrl_dt = mj_model.opt.timestep * physics_steps_per_control_step
GAMMA = 0.97
LAMBDA = 0.95
MINI_BATCHES = 24
ROLLOUT_STEPS = 40
NUM_ENVS = 3072
TOTAL_SAMPLES = ROLLOUT_STEPS * NUM_ENVS
BATCH_SIZE = TOTAL_SAMPLES // MINI_BATCHES

EPOCHS = 8
TOTAL_RUNS = 2_000_000

OBS_DIM = 153
ACTION_DIM = 8
max_global_norm = 0.5
learning_rate = 3e-4
CLIP_EPS = 0.2


class BipeadlEnV:
    def __init__(
            self,
            model,
            forward_weight=1.25,  #1.25
            ctrl_cost_weight=0.1,
            healthy_reward=5.0,  #5.0
            terminate_when_unhealthy=True,
            healthy_z_range=(0.55, 1.3),
            reset_noise_scale=1e-2,
            max_steps=1500
    ):
        self.model = model
        self.forward_weight = forward_weight
        self.ctrl_cost_weight = ctrl_cost_weight
        self.healthy_reward = healthy_reward
        self.terminate_when_unhealthy = terminate_when_unhealthy
        self.healthy_z_range = healthy_z_range
        self.reset_noise_scale = reset_noise_scale
        self.max_steps = max_steps

    def _get_obs(self, data: mjx.Data) -> jnp.ndarray:
        return jnp.concatenate([
            data.qpos[2:], data.qvel, data.cinert[1:].ravel(), data.cvel[1:].ravel(), data.qfrc_actuator
        ])

    def _reset_single(self, rng):
        key1, key2 = jax.random.split(rng, 2)
        qpos = self.model.qpos0 + jax.random.uniform(key1, (self.model.nq,), minval=-1e-2, maxval=1e-2,
                                                     dtype=jnp.float32)
        qvel = jax.random.uniform(key2, (self.model.nv,), minval=-1e-2, maxval=1e-2, dtype=jnp.float32)
        data = mjx.make_data(self.model).replace(qpos=qpos, qvel=qvel)
        data = mjx.forward(self.model, data)
        return data, self._get_obs(data)

    def step_single(self, data: mjx.Data, action: jnp.ndarray, key, step):
        com_before = data.subtree_com[1]
        data = data.replace(ctrl=action)

        data = jax.lax.fori_loop(0, physics_steps_per_control_step, lambda i, d: mjx.step(self.model, d), data)

        com_after = data.subtree_com[1]
        velocity = (com_after - com_before) / ctrl_dt

        forward_reward = self.forward_weight * velocity[0]
        minz, maxz = self.healthy_z_range[0], self.healthy_z_range[1]
        is_healthy = jnp.where((data.qpos[2] >= minz) & (data.qpos[2] <= maxz), 1.0, 0.0)

        reward = forward_reward + (self.healthy_reward * is_healthy) - (
                    self.ctrl_cost_weight * jnp.sum(jnp.square(action)))

        done = 1.0 - is_healthy

        key1, key2 = jax.random.split(key)
        truncated = step > self.max_steps
        data, obs = jax.lax.cond(jnp.logical_or(done == 1.0, truncated), lambda d: self._reset_single(key2),
                                 lambda d: (d, self._get_obs(d)), data)
        step = jnp.where(jnp.logical_or(truncated, done), 0, step + 1)

        return data, obs, reward, done, truncated, key1, step


env = BipeadlEnV(mjx_model)

batched_reset = jax.jit(jax.vmap(env._reset_single, in_axes=(0,)))
batched_step = jax.jit(jax.vmap(env.step_single, in_axes=(0, 0, 0, 0)))


@jax.jit
def gaussian_dist(action, mean, std):
    var = jnp.square(std)
    return -0.5 * (jnp.square(action - mean) / var + 2.0 * jnp.log(std) + jnp.log(2.0 * jnp.pi))


batched_gaussian_dist = jax.vmap(gaussian_dist, in_axes=(0, 0, None))
default_kernel_init = nn.initializers.orthogonal(jnp.sqrt(2))


class ActorCritic(nn.Module):
    action_dim: int

    # state_dim : int

    @nn.compact
    def __call__(self, obs):
        x = nn.Dense(features=256, kernel_init=default_kernel_init)(obs)
        x = nn.relu(x)
        x = nn.Dense(features=256, kernel_init=default_kernel_init)(x)
        x = nn.relu(x)
        value = nn.Dense(features=1, kernel_init=default_kernel_init)(x)
        value = jnp.squeeze(value, axis=-1)

        a = nn.Dense(features=256, kernel_init=default_kernel_init)(obs)
        a = nn.relu(a)
        a = nn.Dense(features=256, kernel_init=default_kernel_init)(a)
        a = nn.relu(a)
        mean = nn.Dense(features=self.action_dim, kernel_init=default_kernel_init)(a)

        log_std = self.param('log_std', lambda rng, shape: jnp.zeros(shape, ), (self.action_dim,))
        log_std = jnp.clip(log_std, -2.0, 0.0)
        sample_key = self.make_rng('noise')
        noise = jax.random.normal(sample_key, mean.shape)
        std = jnp.exp(log_std)
        action = mean + std * noise
        clipped_action = jnp.clip(action, -1.0, 1.0)

        log_prob = jnp.sum(batched_gaussian_dist(action, mean, std), axis=-1)

        return clipped_action, action, value, log_prob, mean, std


def calc_single(i, carry):
    r, m, v, g, adv, lam, lgae, last_value, tr = carry
    i_work = -1 - i
    nv = jax.lax.cond(jnp.equal(i_work, -1), lambda _: last_value, lambda _: v[i_work + 1], operand=None)
    delta = r[i_work] + g * nv * m[i_work] - v[i_work]
    chain = m[i_work] * (1.0 - tr[i_work])
    adv = adv.at[i_work].set(delta + g * lam * chain * lgae)
    lgae = adv[i_work]
    return (r, m, v, g, adv, lam, lgae, last_value, tr)


@jax.jit
def gae_calc(values, rewards, dones, last_value, truncateds):
    gae = jnp.zeros(rewards.shape[0])
    mask = 1.0 - dones
    _, _, _, _, gae, _, _, _, _ = jax.lax.fori_loop(0, rewards.shape[0], calc_single,
                                                    (rewards, mask, values, GAMMA, gae, LAMBDA, 0.0, last_value,
                                                     truncateds))
    return gae, gae + values


batched_gae = jax.vmap(gae_calc, in_axes=(0, 0, 0, 0, 0))
model = ActorCritic(ACTION_DIM)

key = jax.random.PRNGKey(seed=0)
key, init_key, noise_key = jax.random.split(key, 3)
dummy_state = jnp.zeros(shape=(NUM_ENVS, OBS_DIM))

variables = model.init({'params': init_key, 'noise': noise_key}, dummy_state)
params = variables['params']
tx = optax.chain(
    optax.clip_by_global_norm(max_global_norm),
    optax.adam(learning_rate=learning_rate)
)

state = TrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=tx
)


# state = TrainState.create(apply_fn=model.apply, params=params, tx=tx)


def ppo_loss(noise_key, state_params, states, actions, old_log, gae, returns):
    _, _, new_value, _, mean, std = model.apply({'params': state_params}, states, rngs={'noise': noise_key})
    new_log_prob = jnp.sum(batched_gaussian_dist(actions, mean, std), axis=-1)
    ratio = jnp.exp(new_log_prob - old_log)
    surr1 = ratio * gae
    surr2 = jnp.clip(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * gae
    policy_loss = -jnp.mean(jnp.minimum(surr1, surr2))
    entropy = jnp.sum(0.5 + 0.5 * jnp.log(2 * jnp.pi * jnp.square(std)), axis=-1)
    value_loss = 0.5 * jnp.mean(jnp.square(new_value - returns))
    return policy_loss + 0.5 * value_loss - 0.01 * jnp.mean(entropy), (policy_loss, value_loss)


@jax.jit
def update_network(key, state, states_b, actions_b, log_prob_b, gae_b, returns_b):
    gae_b = (gae_b - jnp.mean(gae_b)) / (jnp.std(gae_b) + 1e-8)

    key, noise_key = jax.random.split(key)
    grad_fn = jax.value_and_grad(ppo_loss, has_aux=True, argnums=1)
    (loss, (p_loss, v_loss)), grad = grad_fn(noise_key, state.params, states_b, actions_b, log_prob_b, gae_b, returns_b)
    new_state = state.apply_gradients(grads=grad)
    return new_state, loss


@jax.jit
def make_rollout(state_params, data, obs, key, step):
    observe = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS, OBS_DIM))
    rewards = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS))
    dones = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS))
    truncateds = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS))
    clipped_actions = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS, ACTION_DIM))
    actions = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS, ACTION_DIM))
    values = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS))
    log_probs = jnp.zeros((NUM_ENVS, ROLLOUT_STEPS))

    def to_do(i, carry):
        observe, obs, rewards, dones, truncateds, data, key, step, clipped_actions, actions, values, log_probs = carry
        key, noise, step_key = jax.random.split(key, 3)
        clipped_action, action, value, log_prob, _, _ = model.apply({'params': state_params}, obs,
                                                                    rngs={'noise': noise})

        observe = observe.at[:, i, :].set(obs)

        data, obs, reward, done, truncated, key1, step = batched_step(data, clipped_action,
                                                                      jax.random.split(step_key, NUM_ENVS), step)

        rewards = rewards.at[:, i].set(reward)
        dones = dones.at[:, i].set(done)
        truncateds = truncateds.at[:, i].set(truncated)
        clipped_actions = clipped_actions.at[:, i, :].set(clipped_action)
        actions = actions.at[:, i, :].set(action)
        values = values.at[:, i].set(value)
        log_probs = log_probs.at[:, i].set(log_prob)
        return (observe, obs, rewards, dones, truncateds, data, key, step, clipped_actions, actions, values, log_probs)

    observe, obs, rewards, dones, truncateds, data, key, step, clipped_actions, actions, values, log_probs = jax.lax.fori_loop(
        0, ROLLOUT_STEPS, to_do,
        (observe, obs, rewards, dones, truncateds, data, key, step, clipped_actions, actions, values, log_probs))

    key, noise_key = jax.random.split(key)
    _, _, l_value, _, _, _ = model.apply({'params': state_params}, obs, rngs={'noise': noise_key})
    gae, returns = batched_gae(values, rewards, dones, l_value, truncateds)

    return observe, rewards, dones, truncateds, clipped_actions, actions, values, log_probs, gae, returns, data, obs, key, step


@jax.jit
def train_epochs(key, state, obs_b, actions_b, log_prob_b, gae_b, returns_b):
    obs_f = jnp.reshape(obs_b, (-1, OBS_DIM))
    actions_f = jnp.reshape(actions_b, (-1, ACTION_DIM))
    log_probs_f = jnp.reshape(log_prob_b, (-1,))
    gae_f = jnp.reshape(gae_b, (-1,))
    returns_f = jnp.reshape(returns_b, (-1,))

    def inside_to_do(i, carry):
        key, state, obs_s, actions_s, log_probs_s, gae_s, returns_s = carry
        key, per_key = jax.random.split(key)
        state, _ = update_network(per_key, state, obs_s[i], actions_s[i], log_probs_s[i], gae_s[i], returns_s[i])

        return (key, state, obs_s, actions_s, log_probs_s, gae_s, returns_s)

    def to_do(i, carry):
        key, state, obs_f, actions_f, log_probs_f, gae_f, returns_f = carry
        key, per_key = jax.random.split(key)

        permutation = jax.random.permutation(per_key, TOTAL_SAMPLES)
        idx = permutation.reshape(MINI_BATCHES, BATCH_SIZE)
        obs_s = obs_f[idx]
        actions_s = actions_f[idx]
        log_probs_s = log_probs_f[idx]
        gae_s = gae_f[idx]
        returns_s = returns_f[idx]

        key, state, _, _, _, _, _ = jax.lax.fori_loop(0, MINI_BATCHES, inside_to_do,
                                                      (key, state, obs_s, actions_s, log_probs_s, gae_s, returns_s))

        return (key, state, obs_f, actions_f, log_probs_f, gae_f, returns_f)

    key, state, _, _, _, _, _ = jax.lax.fori_loop(0, EPOCHS, to_do,
                                                  (key, state, obs_f, actions_f, log_probs_f, gae_f, returns_f))
    return key, state


key, reset_key = jax.random.split(key)
data, obs = batched_reset(jax.random.split(reset_key, NUM_ENVS))
step = jnp.zeros((NUM_ENVS,), dtype=jnp.int32)
LOG_EVERY = 25
SAVE_EVERY = 50
CHECKPOINT_PATH = 'biped_checkpoint.pkl'


def save_checkpoint(path, params, opt_state, iteration):
    # write to a temp file first so a crash mid-save can't corrupt the old checkpoint
    with open(path + '.tmp', 'wb') as f:
        pickle.dump({'params': jax.device_get(params),
                     'opt_state': jax.device_get(opt_state),
                     'iteration': int(iteration)}, f)
    os.replace(path + '.tmp', path)


def load_checkpoint(path, state):
    with open(path, 'rb') as f:
        ck = pickle.load(f)
    state = state.replace(params=jax.tree_util.tree_map(jnp.asarray, ck['params']),
                          opt_state=jax.tree_util.tree_map(jnp.asarray, ck['opt_state']))
    return state, ck['iteration']


# state, start_iter = load_checkpoint('biped_checkpoint.pkl', state)
@jax.jit
def total_run(state, data, obs, key, step):
    def to_do(i, carry):
        state, data, obs, key, step = carry

        observe, rewards, dones, truncateds, clipped_actions, actions, values, log_probs, gae, returns, data, obs, key, step = make_rollout(
            state.params, data, obs, key, step)

        key, state = train_epochs(key, state, observe, actions, log_probs, gae, returns)

        n_end = jnp.sum(jnp.maximum(dones, truncateds))
        jax.lax.cond(
            i % LOG_EVERY == 0,
            lambda: jax.debug.print("iter {i} | mean reward {r:.3f} | episodes ended {n} avg ep length {m}", i=i,
                                    r=jnp.mean(rewards), n=n_end, m=TOTAL_SAMPLES / jnp.maximum(n_end, 1)),
            lambda: None,
        )
        jax.lax.cond(
            i % SAVE_EVERY == 0,
            lambda: jax.experimental.io_callback(lambda p, o, it: save_checkpoint(CHECKPOINT_PATH, p, o, it), None,
                                                 state.params, state.opt_state, i, ordered=True),
            lambda: None,
        )

        return (state, data, obs, key, step)

    state, data, obs, key, step = jax.lax.fori_loop(0, TOTAL_RUNS, to_do, (state, data, obs, key, step))
    return (state, data, obs, key, step)


out = total_run(state, data, obs, key, step)
# save_checkpoint(CHECKPOINT_PATH, state.params, state.opt_state, TOTAL_RUNS - 1)